# LangChain Internals — Build It From Scratch

## Why build LangChain from scratch?

The best way to understand a framework is to build a simplified version of it yourself. This notebook implements the core concepts behind LangChain using only plain Python — no LangChain imports.

By doing this, you'll see that LangChain is not magic. It's a well-designed wrapper around a few simple ideas:

```
Template fills in variables → produces a string
LLM takes a string → produces a response
Parser takes a response → extracts the useful part
```

LangChain just provides:
1. Consistent interfaces for all of the above (so you can swap providers easily)
2. The `|` pipe operator to chain them together
3. Extras like streaming, batching, caching, callbacks, etc.

## What this notebook builds

### `MyLLM`
A mock LLM class with a `.predict(prompt)` method that returns a random canned response. This shows what a real LLM wrapper does — it takes a string prompt and returns a response.

### `MyPromptTemplate`
A simple template class with a `.format(input_dict)` method that fills in variables. This is exactly what LangChain's `PromptTemplate` does under the hood.

## The "Runnable" concept

In LangChain, everything has a `.invoke()` method:
- `PromptTemplate.invoke({"topic": "AI"})` → fills the template
- `ChatOllama.invoke(prompt)` → calls the model
- `StrOutputParser.invoke(response)` → extracts the text

They all follow the same interface, which is why you can chain them with `|`. When you write `chain = prompt | model | parser`, Python's `__or__` operator is intercepted by LangChain to create a composed Runnable.

## What you'll learn

- The minimal code needed to replicate LangChain's core pattern
- Why the `|` operator works between LangChain components
- What `.invoke()` is really doing at each step
- How to build your own custom Runnable steps

## Prerequisites

- No API key or Ollama needed — this notebook uses only Python builtins
- Virtual environment activated

In [1]:
import random

class MyLLM:
    
    def __init__(self):
        print("MyLLM initialized")
        
    def predict(self, prompt):
        
        response_list = [
            "Some response based on the prompt",
            "Another response based on the prompt",
            "Yet another response based on the prompt"
        ]
        
        
        return {"response": random.choice(response_list)}

In [2]:
llm = MyLLM()
response = llm.predict("What is the meaning of life?")
print(response)

MyLLM initialized
{'response': 'Another response based on the prompt'}


In [3]:
llm.predict("What is the meaning of life?")

{'response': 'Some response based on the prompt'}

In [5]:
class MyPromptTemplate:
    
    def __init__(self, template,input_variables):
        self.template = template
        self.input_variables = input_variables

    def format(self, input_dict):
        return self.template.format(**input_dict)

In [6]:
template = MyPromptTemplate(template="What is the meaning of {topic}?", input_variables= ["topic"])

In [8]:
template.format({"topic": "life"})

'What is the meaning of life?'